In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Define project paths
BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR
OUTPUT_DIR = BASE_DIR / "cleaned_data"
OUTPUT_DIR.mkdir(exist_ok=True)

# Load datasets
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_details = pd.read_csv(DATA_DIR / "order_details.csv")
pizzas = pd.read_csv(DATA_DIR / "pizzas.csv")
pizza_types = pd.read_csv(DATA_DIR / "pizza_types.csv")

# Store datasets for quality checks
datasets = {
    "orders": orders,
    "order_details": order_details,
    "pizzas": pizzas,
    "pizza_types": pizza_types
}

# Inspect data quality
for name, data in datasets.items():
    print(f"\n{name.upper()} DATA")
    print("Shape:", data.shape)
    print("\nMissing values:\n", data.isnull().sum())
    print("Duplicate rows:", data.duplicated().sum())
    print("\nData types:\n", data.dtypes)

# Make copies before cleaning
orders = orders.copy()
order_details = order_details.copy()
pizzas = pizzas.copy()
pizza_types = pizza_types.copy()

# Remove extra spaces in text columns
for data in [orders, order_details, pizzas, pizza_types]:
    text_columns = data.select_dtypes(include="object").columns
    for col in text_columns:
        data[col] = data[col].str.strip()

# Convert date and time columns
if "date" in orders.columns:
    orders["date"] = pd.to_datetime(
        orders["date"], errors="coerce"
    )

if "time" in orders.columns:
    orders["time"] = pd.to_datetime(
        orders["time"], format="%H:%M:%S", errors="coerce"
    ).dt.time

# Remove exact duplicate records
orders = orders.drop_duplicates()
order_details = order_details.drop_duplicates()
pizzas = pizzas.drop_duplicates()
pizza_types = pizza_types.drop_duplicates()

# Convert numeric columns where applicable
for data, columns in [
    (orders, ["order_id"]),
    (order_details, ["order_id", "quantity"]),
    (pizzas, ["price"]),
    (pizza_types, ["pizza_type_id"])
]:
    for col in columns:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

# Check missing values after cleaning
cleaned_datasets = {
    "orders": orders,
    "order_details": order_details,
    "pizzas": pizzas,
    "pizza_types": pizza_types
}

for name, data in cleaned_datasets.items():
    print(f"\n{name.upper()} AFTER CLEANING")
    print("Shape:", data.shape)
    print("Missing values:\n", data.isnull().sum())
    print("Duplicates:", data.duplicated().sum())

# Export cleaned datasets
for name, data in cleaned_datasets.items():
    data.to_csv(OUTPUT_DIR / f"{name}_cleaned.csv", index=False)

print("\nCleaning process completed.")